# 01 · The Update Is Low-Rank

The fact that makes LoRA possible. Fully fine-tune a small model, subtract the
original weights, and take the SVD of the update $\Delta W = W_{new} - W_{old}$.
Its singular values **collapse** — the change the task made lives in a few
directions. This is the same low-rank structure PCA/SVD find in *data*
(`dim_reduction_arena`), here found in the *update*. Before any adapter, there is
far less to learn than the parameter count implies.

In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))
print('repo root:', _p)

repo root: /workspaces/lora_101


In [2]:
import torch
from src.config import get_settings
from src.tasks.toy import make_toy_task, make_base_model
from src.finetune.full import make_full_finetune
from src.finetune.train import train
from src.analysis.spectrum import update_delta, update_spectrum, effective_rank
from src.analysis.subspace import captured_energy
from src.visualisation import plot_spectrum, save_figure
from src.results import save_json

s = get_settings()
torch.manual_seed(s.seed)
task = make_toy_task(dim=s.toy.dim, out_dim=s.toy.out_dim, true_rank=s.toy.true_rank,
                     n_samples=s.toy.n_samples, seed=s.seed)
print('weight shape:', tuple(task.w_base.shape), '| planted update rank:', task.true_rank)

weight shape: (128, 128) | planted update rank: 4


## Fully fine-tune, then look at what moved
Train every weight on the new task, then inspect the update.

In [3]:
base = make_base_model(task)
full = make_full_finetune(base)
hist = train(full, task.inputs, task.targets, steps=s.train.steps, lr=s.train.lr, seed=s.seed)
w_old = task.w_base
w_new = full.linear.weight.detach()
delta = update_delta(w_old, w_new)
sv = update_spectrum(w_old, w_new)
er = effective_rank(sv, 0.99)
print(f'full fine-tune loss: {hist[0]:.3e} -> {hist[-1]:.3e}')
print('top-8 singular values:', [round(float(x), 3) for x in sv[:8]])
print(f'effective rank (99% energy): {er}  (planted true_rank={task.true_rank})')

full fine-tune loss: 1.001e+00 -> 2.542e-05
top-8 singular values: [6.379, 5.754, 5.448, 4.952, 0.021, 0.015, 0.014, 0.011]
effective rank (99% energy): 4  (planted true_rank=4)


## Save the spectrum (PNG) and the numbers (JSON)

In [4]:
k = min(len(sv), 2 * task.true_rank + 8)
fig = plot_spectrum(sv[:k].tolist(), title='Full fine-tune update spectrum (it collapses)')
png = save_figure(fig, '01_update_spectrum')
json_path = save_json({
    'planted_true_rank': task.true_rank,
    'effective_rank_99': er,
    'singular_values': [float(x) for x in sv],
    'captured_energy_by_rank': {str(r): captured_energy(delta, r) for r in [1, 2, 4, 8, 16]},
    'full_finetune_loss_start': hist[0],
    'full_finetune_loss_end': hist[-1],
    'weight_shape': list(task.w_base.shape),
}, '01_the_update_is_low_rank')
print('saved:', png.name, 'and', json_path.name)

saved: 01_update_spectrum.png and 01_the_update_is_low_rank.json


## Takeaway
The update is low-rank: a handful of nonzero singular values, the rest ~0, and the
effective rank equals the task's true rank. LoRA will learn *those directions*
directly instead of the full matrix around them — the subject of notebook 02.